In [1]:
import os
import pandas as pd
import taxonomia

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [2]:
def obtener_paths_cantos(base_dir):
    data = []

    # Recorrer las familias
    for familia in os.listdir(base_dir):
        if familia == '.DS_Store':
            continue
        familia_path = os.path.join(base_dir, familia)
        if os.path.isdir(familia_path):
            # Recorrer los géneros
            for genero in os.listdir(familia_path):
                if genero == '.DS_Store':
                    continue
                genero_path = os.path.join(familia_path, genero)
                if os.path.isdir(genero_path):
                    # Recorrer las especies
                    for especie in os.listdir(genero_path):
                        if especie == '.DS_Store':
                            continue
                        especie_path = os.path.join(genero_path, especie)
                        if os.path.isdir(especie_path):
                            # Recorrer los nombres comunes
                            for nombre_comun in os.listdir(especie_path):
                                if nombre_comun == '.DS_Store':
                                    continue
                                nombre_comun_path = os.path.join(especie_path, nombre_comun)
                                if os.path.isdir(nombre_comun_path):
                                    # Recorrer los cantos
                                    for canto in os.listdir(nombre_comun_path):
                                        if canto == '.DS_Store':
                                            continue
                                        canto_path = os.path.join(nombre_comun_path, canto)
                                        if os.path.isfile(canto_path):
                                            data.append({
                                                'Specie': especie,
                                                'Genus': genero,
                                                'Family': familia,
                                                'audio_path': canto_path
                                            })

    # Crear el DataFrame
    df = pd.DataFrame(data)
    return df

# Definir la ruta base
base_dir = '/Users/camcortes/Documents/birds-sounds/notebooks/songs'

# Obtener el DataFrame
df = obtener_paths_cantos(base_dir)

# Mostrar el DataFrame
df.sample(4)

,Specie,Genus,Family,audio_path
67861,Asemospiza fuliginosa,Asemospiza,Thraupidae,/Users/camcortes/Documents/birds-sounds/notebo...
67902,Asemospiza fuliginosa,Asemospiza,Thraupidae,/Users/camcortes/Documents/birds-sounds/notebo...
64990,Melanerpes formicivorus,Melanerpes,Picidae,/Users/camcortes/Documents/birds-sounds/notebo...
56886,Pheugopedius coraya,Pheugopedius,Troglodytidae,/Users/camcortes/Documents/birds-sounds/notebo...


In [3]:
print(df['audio_path'][0])

/Users/camcortes/Documents/birds-sounds/notebooks/songs/Capitonidae/Capito/Capito auratus/GildedBarbet/47717.mp3


In [4]:
df.shape

(80714, 4)

In [5]:
familia = taxonomia.capitalize_family_genus(taxonomia.genus_family)
df['Family'] = df['Genus'].map(familia)
df['Suborder'] = df['Family'].map(taxonomia.suborder)

In [6]:
df.head()

,Specie,Genus,Family,audio_path,Suborder
0,Capito auratus,Capito,Capitonidae,/Users/camcortes/Documents/birds-sounds/notebo...,Piciformes
1,Capito auratus,Capito,Capitonidae,/Users/camcortes/Documents/birds-sounds/notebo...,Piciformes
2,Capito auratus,Capito,Capitonidae,/Users/camcortes/Documents/birds-sounds/notebo...,Piciformes
3,Capito auratus,Capito,Capitonidae,/Users/camcortes/Documents/birds-sounds/notebo...,Piciformes
4,Capito auratus,Capito,Capitonidae,/Users/camcortes/Documents/birds-sounds/notebo...,Piciformes


In [7]:
#filtrar las especies que tengan mas de 30 cantos
df = df.groupby('Specie').filter(lambda x: len(x) >= 30)
df['Suborder'].value_counts()

Suborder
tiranni           42725
passeri           31638
Falconiformes       607
Piciformes          541
Psittaciformes      225
Name: count, dtype: int64

In [8]:
df = df.rename(columns={'Specie':'label', 'Path': 'audio_path'})
df = df.reset_index(drop=True)

In [9]:
df.shape

(75736, 5)

In [10]:
from sklearn.model_selection import StratifiedShuffleSplit

X = df.drop(columns=['label'])
y = df['label']

sss = StratifiedShuffleSplit(n_splits=1, test_size=0.3, random_state=42)

for train_index, test_index in sss.split(X, y):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y[train_index], y[test_index]

In [11]:
X_train['label'] = y_train
X_test['label'] = y_test

/var/folders/_1/9ctwy_pn6bv2hyw11l_gqhm9855lbl/T/ipykernel_47590/2141970401.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train['label'] = y_train
/var/folders/_1/9ctwy_pn6bv2hyw11l_gqhm9855lbl/T/ipykernel_47590/2141970401.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_test['label'] = y_test


In [12]:
# guardar el dataframe en un archivo csv
df.to_csv('/Users/camcortes/Documents/birds-sounds/data/paths_cantos_passeriformes.csv', index=False)
X_train.to_csv('/Users/camcortes/Documents/birds-sounds/data/train_specie.csv', index=False)
X_test.to_csv('/Users/camcortes/Documents/birds-sounds/data/test_specie.csv', index=False)

# muestra del df
muestra = X_train.sample(2000, random_state=42)
muestra = muestra.groupby('label').filter(lambda x: len(x) >= 5)

In [13]:
# split de los datos en train y test para el modelo 
from sklearn.model_selection import train_test_split
X_train_muestra, X_test_muestra = train_test_split(
    muestra, 
    test_size=0.3, 
    random_state=42,
    stratify=muestra['label']
    )

X_train_muestra.to_csv('/Users/camcortes/Documents/birds-sounds/data/train_muestra.csv', index=False)
X_test_muestra.to_csv('/Users/camcortes/Documents/birds-sounds/data/test_muestra.csv', index=False)